                             RAG
                              │
          ┌───────────────────┼───────────────────┐
          │                   │                   │
   OpenAIEmbeddings     Vector Store         Retriever
                                                  │
                                                  ▼
                                          Retrieved Docs
                                                  │
                                                  ▼
                                      ChatPromptTemplate
                                                  │
                                                  ▼
                                           ChatOpenAI
                                                  │
                                                  ▼
                                        StrOutputParser
                                                  │
                                                  ▼
                                             Response
                                                  │
                                                  │
                         RAGAS EVALUATION          │
                              │                   │
                              ▼                   ▼
                         Dataset              Response
                              │                   │
                              └─────────┬─────────┘
                                        ▼
                                   @experiment
                                        │
                         ┌──────────────┴─────────────┐
                         │                            │
                 DiscreteMetric                NumericMetric
                  Correctness                   Relevance
                         │                            │
                         └──────────────┬─────────────┘
                                        ▼
                                  Score + Reason
                                        │
                                        ▼
                                Experiment Results
                                        │
                                        ▼
                                      CSV

In [1]:

#
# Install:
#
# pip install -U \
#     ragas \
#     langchain \
#     langchain-openai \
#     openai \
#     pandas \
#     nest_asyncio
#
# Run:
#
# python ragas_langchain_eval.py
#
# ============================================================


# ============================================================
# 1. IMPORTS
# ============================================================

import os
import asyncio
import getpass
from pathlib import Path
import pandas as pd

In [2]:
# ------------------------------------------------------------
# LANGCHAIN IMPORTS
# ------------------------------------------------------------
from langchain_openai import (
    ChatOpenAI,
    OpenAIEmbeddings,
)

from langchain_core.documents import Document

from langchain_core.prompts import (
    ChatPromptTemplate,
)

from langchain_core.output_parsers import (
    StrOutputParser,
)

from langchain_core.vectorstores import (
    InMemoryVectorStore,
)

In [4]:
# ------------------------------------------------------------
# RAGAS IMPORTS
# ------------------------------------------------------------
from tqdm.auto import tqdm

from ragas import (
    Dataset,
    experiment,
)

from ragas.llms import (
    llm_factory,
)

from ragas.metrics import (
    DiscreteMetric,
    NumericMetric,
)

In [5]:
# ------------------------------------------------------------
# OPENAI CLIENT
#
# NOTE:
# This client is ONLY for the RAGAS evaluator.
#
# The actual RAG application uses LangChain.
# ------------------------------------------------------------

from openai import AsyncOpenAI


In [7]:
# ============================================================
# 2. API KEY
# ============================================================

import os
from dotenv import load_dotenv

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if not OPENAI_API_KEY:
    raise ValueError("OPENAI_API_KEY not found in .env")

print("API key loaded successfully")

API key loaded successfully


In [8]:
# ============================================================
# 3. MODEL CONFIGURATION
# ============================================================

APP_MODEL = os.getenv(
    "APP_MODEL",
    "gpt-4o-mini",
)

EMBEDDING_MODEL = os.getenv(
    "EMBEDDING_MODEL",
    "text-embedding-3-small",
)

JUDGE_MODEL = os.getenv(
    "JUDGE_MODEL",
    "gpt-4o-mini",
)

TOP_K = 2


print(
    "Application Model:",
    APP_MODEL,
)

print(
    "Embedding Model:",
    EMBEDDING_MODEL,
)

print(
    "Judge Model:",
    JUDGE_MODEL,
)

print(
    "Top K:",
    TOP_K,
)


Application Model: gpt-4o-mini
Embedding Model: text-embedding-3-small
Judge Model: gpt-4o-mini
Top K: 2


In [9]:
# ============================================================
# 4. LANGCHAIN APPLICATION LLM
# ============================================================

rag_llm = ChatOpenAI(
    model=APP_MODEL,
    temperature=0,
)


# ============================================================
# 5. LANGCHAIN EMBEDDING MODEL
# ============================================================

embeddings = OpenAIEmbeddings(
    model=EMBEDDING_MODEL,
)

In [11]:
# ============================================================
# 6. KNOWLEDGE BASE
# ============================================================

DOCUMENTS = [

    Document(
        page_content="""
Ragas is an open-source evaluation framework designed
for evaluating and improving LLM applications.

It supports RAG evaluation, agent evaluation,
custom metrics, datasets, experiments,
and synthetic test data generation.
""".strip(),
        metadata={
            "doc_id": "ragas_intro",
        },
    ),

    Document(
        page_content="""
Ragas experiments allow developers to systematically
evaluate AI applications, track evaluation results,
compare different versions, and store experiment outputs.
""".strip(),
        metadata={
            "doc_id": "ragas_experiments",
        },
    ),

    Document(
        page_content="""
Ragas metrics measure different properties of AI systems.

Metrics can evaluate factual correctness,
faithfulness, answer relevancy,
retrieval quality, agent behavior,
and application-specific requirements.
""".strip(),
        metadata={
            "doc_id": "ragas_metrics",
        },
    ),

    Document(
        page_content="""
RAG stands for Retrieval-Augmented Generation.

A RAG system retrieves relevant information
from a knowledge base and provides that information
to an LLM for answer generation.
""".strip(),
        metadata={
            "doc_id": "rag_intro",
        },
    ),

]


print(
    "\nKnowledge Base Documents:",
    len(DOCUMENTS),
)


Knowledge Base Documents: 4


In [12]:
# ============================================================
# 7. LANGCHAIN VECTOR STORE
# ============================================================

vector_store = InMemoryVectorStore(
    embedding=embeddings,
)

vector_store.add_documents(
    DOCUMENTS
)

print(
    "Documents added to vector store."
)

Documents added to vector store.


In [13]:
# ============================================================
# 8. LANGCHAIN RETRIEVER
# ============================================================

retriever = vector_store.as_retriever(

    search_kwargs={
        "k": TOP_K,
    }

)

In [14]:
# ============================================================
# 9. LANGCHAIN PROMPT
# ============================================================

rag_prompt = ChatPromptTemplate.from_template(
    """
You are a helpful AI assistant.

Answer the user's question ONLY using
the supplied context.

Rules:

1. Use only the provided context.
2. Do not invent information.
3. Do not use outside knowledge.
4. If the answer is not present in the context,
   say:

   "I don't know based on the provided context."

5. Keep the answer concise and factual.


CONTEXT:

{context}


QUESTION:

{question}


ANSWER:
"""
)

In [15]:
# ============================================================
# 10. LANGCHAIN RAG GENERATION CHAIN
# ============================================================

rag_chain = (

    rag_prompt

    |

    rag_llm

    |

    StrOutputParser()

)

# ============================================================
# 11. HELPER FUNCTION TO FORMAT DOCUMENTS
# ============================================================

def format_documents(
    documents,
):

    return "\n\n".join(

        document.page_content

        for document in documents

    )

In [16]:
# ============================================================
# 12. COMPLETE LANGCHAIN RAG APPLICATION
# ============================================================

async def rag_query(
    question: str,
):

    # --------------------------------------------------------
    # STEP 1
    # Retrieve relevant documents using LangChain Retriever
    # --------------------------------------------------------

    relevant_documents = await retriever.ainvoke(
        question
    )


    # --------------------------------------------------------
    # STEP 2
    # Create context
    # --------------------------------------------------------

    context = format_documents(
        relevant_documents
    )


    # --------------------------------------------------------
    # STEP 3
    # Generate answer using LangChain LCEL chain
    # --------------------------------------------------------

    answer = await rag_chain.ainvoke(
        {
            "context": context,
            "question": question,
        }
    )


    # --------------------------------------------------------
    # STEP 4
    # Capture retrieved contexts
    # --------------------------------------------------------

    retrieved_contexts = [

        document.page_content

        for document
        in relevant_documents

    ]


    # --------------------------------------------------------
    # STEP 5
    # Capture document IDs
    # --------------------------------------------------------

    retrieved_document_ids = [

        document.metadata.get(
            "doc_id"
        )

        for document
        in relevant_documents

    ]


    return {

        "answer":
            answer,

        "retrieved_contexts":
            retrieved_contexts,

        "retrieved_document_ids":
            retrieved_document_ids,

    }

In [17]:
# ============================================================
# 13. RAGAS EVALUATOR LLM
# ============================================================
#
# IMPORTANT:
#
# Current RAGAS recommends llm_factory().
#
# LangchainLLMWrapper exists in older examples,
# but current RAGAS migration docs classify that
# path as deprecated.
#
# ============================================================

ragas_openai_client = AsyncOpenAI(

    api_key=os.environ[
        "OPENAI_API_KEY"
    ],

    timeout=60.0,

    max_retries=3,

)


evaluator_llm = llm_factory(

    JUDGE_MODEL,

    client=ragas_openai_client,

)


In [18]:
# ============================================================
# 14. CREATE RAGAS EVALUATION DATASET
# ============================================================

def load_dataset():

    Path(
        "evals"
    ).mkdir(
        exist_ok=True
    )


    dataset = Dataset(

        name="ragas_langchain_dataset",

        backend="local/csv",

        root_dir="evals",

    )


    data_samples = [

        {

            "question":
                "What is Ragas?",

            "grading_notes":
                """
Ragas is an evaluation framework
for LLM applications.
""".strip(),

        },


        {

            "question":
                "How do Ragas experiments work?",

            "grading_notes":
                """
Ragas experiments provide systematic evaluation,
track results, compare different application versions,
and store experiment outputs.
""".strip(),

        },


        {

            "question":
                "What do Ragas metrics do?",

            "grading_notes":
                """
Ragas metrics evaluate properties and quality
of AI or LLM application outputs.
""".strip(),

        },


        {

            "question":
                "What is RAG?",

            "grading_notes":
                """
RAG stands for Retrieval-Augmented Generation.

It retrieves relevant information from a knowledge
source and provides that context to an LLM
before answer generation.
""".strip(),

        },

    ]


    for sample in data_samples:

        dataset.append(
            sample
        )


    dataset.save()


    return dataset


In [19]:
# ============================================================
# 15. RAGAS CUSTOM METRIC - CORRECTNESS
# ============================================================
#
# Discrete scoring:
#
# pass
# fail
#
# ============================================================

correctness_metric = DiscreteMetric(

    name="correctness",

    prompt="""
Evaluate whether the model response correctly answers
the user's question according to the grading notes.

QUESTION:

{question}


EXPECTED / GRADING NOTES:

{grading_notes}


MODEL RESPONSE:

{response}


Evaluation Rules:

PASS if:

1. The response contains the important information
   from the grading notes.

2. The response is factually correct.

3. The response answers the user's question.


FAIL if:

1. Important information is missing.

2. The response contradicts the grading notes.

3. The response does not answer the question.


Return only:

pass

or

fail
""",

    allowed_values=[
        "pass",
        "fail",
    ],

)


In [20]:


# ============================================================
# 16. RAGAS CUSTOM METRIC - RELEVANCE
# ============================================================
#
# Numeric score:
#
# 1 = irrelevant
# 5 = completely relevant
#
# ============================================================

relevance_metric = NumericMetric(

    name="relevance",

    prompt="""
Evaluate how relevant the model response is
to the user's question.


QUESTION:

{question}


MODEL RESPONSE:

{response}


Use this rubric:

1 = Completely irrelevant

2 = Mostly irrelevant

3 = Partially relevant

4 = Mostly relevant

5 = Completely relevant


Return only the numerical score.
""",

    allowed_values=(
        1,
        5,
    ),

)

In [21]:
# ============================================================
# 17. RAGAS EXPERIMENT
# ============================================================

@experiment()
async def run_rag_evaluation(
    row,
):

    # --------------------------------------------------------
    # Get question
    # --------------------------------------------------------

    question = row[
        "question"
    ]


    # --------------------------------------------------------
    # Get reference / grading notes
    # --------------------------------------------------------

    grading_notes = row[
        "grading_notes"
    ]


    # ========================================================
    # RUN LANGCHAIN RAG
    # ========================================================

    rag_result = await rag_query(
        question
    )


    response = rag_result[
        "answer"
    ]


    retrieved_contexts = rag_result[
        "retrieved_contexts"
    ]


    retrieved_document_ids = rag_result[
        "retrieved_document_ids"
    ]


    # ========================================================
    # RAGAS CORRECTNESS EVALUATION
    # ========================================================

    correctness_result = (
        await correctness_metric.ascore(

            llm=evaluator_llm,

            question=question,

            grading_notes=grading_notes,

            response=response,

        )
    )


    # ========================================================
    # RAGAS RELEVANCE EVALUATION
    # ========================================================

    relevance_result = (
        await relevance_metric.ascore(

            llm=evaluator_llm,

            question=question,

            response=response,

        )
    )


    # ========================================================
    # RETURN EXPERIMENT DATA
    # ========================================================

    return {

        **row,


        # ----------------------------------------------------
        # Application Output
        # ----------------------------------------------------

        "response":
            response,


        # ----------------------------------------------------
        # Retrieval Information
        # ----------------------------------------------------

        "retrieved_contexts":
            retrieved_contexts,


        "retrieved_document_ids":
            retrieved_document_ids,


        # ----------------------------------------------------
        # Correctness
        # ----------------------------------------------------

        "correctness":
            correctness_result.value,


        "correctness_reason":
            correctness_result.reason,


        # ----------------------------------------------------
        # Relevance
        # ----------------------------------------------------

        "relevance":
            relevance_result.value,


        "relevance_reason":
            relevance_result.reason,

    }


In [22]:
# ============================================================
# 18. MAIN FUNCTION
# ============================================================

async def main():

    print(
        "\n"
        + "=" * 100
    )

    print(
        "RAGAS + LANGCHAIN EVALUATION"
    )

    print(
        "=" * 100
    )


    # ========================================================
    # LOAD DATASET
    # ========================================================

    dataset = load_dataset()


    print(
        "\nDataset loaded."
    )


    # ========================================================
    # RUN RAGAS EXPERIMENT
    # ========================================================

    results = await run_rag_evaluation.arun(

        dataset,

        name="ragas_langchain_experiment",

    )


    # ========================================================
    # CONVERT RESULTS TO PANDAS
    # ========================================================

    df = results.to_pandas()


    # ========================================================
    # DETAILED RESULTS
    # ========================================================

    print(
        "\n"
        + "=" * 100
    )

    print(
        "DETAILED RESULTS"
    )

    print(
        "=" * 100
    )


    columns_to_show = [

        "question",

        "response",

        "correctness",

        "relevance",

    ]


    print(

        df[
            columns_to_show
        ].to_string(
            index=False
        )

    )


    # ========================================================
    # CORRECTNESS PASS RATE
    # ========================================================

    correctness_pass_rate = (

        df[
            "correctness"
        ]

        .astype(str)

        .str.lower()

        .eq(
            "pass"
        )

        .mean()

    )


    # ========================================================
    # AVERAGE RELEVANCE
    # ========================================================

    average_relevance = (

        pd.to_numeric(

            df[
                "relevance"
            ],

            errors="coerce",

        )

        .mean()

    )


    # ========================================================
    # SUMMARY
    # ========================================================

    print(
        "\n"
        + "=" * 100
    )

    print(
        "SUMMARY"
    )

    print(
        "=" * 100
    )


    print(

        f"Correctness Pass Rate : "
        f"{correctness_pass_rate:.1%}"

    )


    print(

        f"Average Relevance     : "
        f"{average_relevance:.2f}/5"

    )


    # ========================================================
    # SAVE RESULTS
    # ========================================================

    experiment_directory = Path(
        "evals/experiments"
    )


    experiment_directory.mkdir(

        parents=True,

        exist_ok=True,

    )


    output_file = (

        experiment_directory

        /

        "ragas_langchain_results.csv"

    )


    df.to_csv(

        output_file,

        index=False,

    )


    print(
        "\nResults saved to:"
    )


    print(
        output_file
    )


    # ========================================================
    # PRINT DETAILED REASONS
    # ========================================================

    print(
        "\n"
        + "=" * 100
    )

    print(
        "DETAILED EVALUATION REASONS"
    )

    print(
        "=" * 100
    )


    for _, result_row in df.iterrows():

        print(
            "\nQUESTION:"
        )

        print(
            result_row[
                "question"
            ]
        )


        print(
            "\nRESPONSE:"
        )

        print(
            result_row[
                "response"
            ]
        )


        print(
            "\nCORRECTNESS:"
        )

        print(
            result_row[
                "correctness"
            ]
        )


        print(
            "\nCORRECTNESS REASON:"
        )

        print(
            result_row[
                "correctness_reason"
            ]
        )


        print(
            "\nRELEVANCE:"
        )

        print(
            result_row[
                "relevance"
            ]
        )


        print(
            "\nRELEVANCE REASON:"
        )

        print(
            result_row[
                "relevance_reason"
            ]
        )


        print(
            "\nRETRIEVED DOCUMENT IDS:"
        )

        print(
            result_row[
                "retrieved_document_ids"
            ]
        )


        print(
            "\n"
            + "-" * 100
        )


In [23]:
# ============================================================
# 19. RUN
# ============================================================
try:
    asyncio.run(
        main()
    )
except RuntimeError as error:

    # ----------------------------------------------------
    # Jupyter Notebook fallback
    # ----------------------------------------------------

    if (
        "asyncio.run() cannot be called"
        in str(error)
        or
        "event loop"
        in str(error).lower()
    ):

        import nest_asyncio

        nest_asyncio.apply()


        loop = asyncio.get_event_loop()


        loop.run_until_complete(
            main()
        )


    else:

        raise


RAGAS + LANGCHAIN EVALUATION

Dataset loaded.


Running experiment: 100%|██████████| 4/4 [00:20<00:00,  5.17s/it]


DETAILED RESULTS
                      question                                                                                                                                                                                           response correctness  relevance
                What is Ragas?                                                                                               Ragas is an open-source evaluation framework designed for evaluating and improving LLM applications.        pass        3.0
     What do Ragas metrics do? Ragas metrics measure different properties of AI systems, including factual correctness, faithfulness, answer relevancy, retrieval quality, agent behavior, and application-specific requirements.        pass        5.0
How do Ragas experiments work?                                 Ragas experiments allow developers to systematically evaluate AI applications, track evaluation results, compare different versions, and store experiment outputs.        pa


C:\Users\Sunny\AppData\Local\Temp\ipykernel_43720\4046066728.py:33: RuntimeWarning: coroutine 'main' was never awaited
  loop.run_until_complete(
